In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It installs MDAnalysis and sets up a sample trajectory
# using MDAnalysis's built-in test data (adenylate kinase, GROMACS format).
#
# This replaces the "peptide.tpr / peptide.xtc" files that would be used in a
# local setup with your own simulation data. The built-in AdK trajectory covers
# the same analysis concepts (RMSD, RMSF, Rg, hydrogen bonds, etc.).
#
# numpy and matplotlib are pre-installed in Google Colab.
# MDAnalysis installation takes ~60 seconds on first run.

import sys, os, subprocess, shutil, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.system()}")

# ── Install MDAnalysis ────────────────────────────────────────────────────────
print("\nInstalling MDAnalysis (this may take ~60 s)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "MDAnalysis", "MDAnalysisTests"])
print("MDAnalysis installed ✓")

# ── Set up built-in test trajectory ──────────────────────────────────────────
# MDAnalysis ships with a 10-frame GROMACS trajectory of adenylate kinase (AdK)
# solved in TIP4P water with the OPLS/AA force field.  It is a fully realistic
# MD system and exercises exactly the same analysis code as a peptide simulation.
from MDAnalysis.tests.datafiles import GRO as _TOPOLOGY_SRC, XTC as _TRAJ_SRC

shutil.copy(_TOPOLOGY_SRC, "peptide.gro")   # topology  (GRO format)
shutil.copy(_TRAJ_SRC,     "peptide.xtc")   # trajectory (XTC format)

print(f"Topology   → peptide.gro  (source: {os.path.basename(_TOPOLOGY_SRC)})")
print(f"Trajectory → peptide.xtc  (source: {os.path.basename(_TRAJ_SRC)})")
print("\nNote: Using adenylate kinase (AdK) as the example system.")
print("To analyse your own simulation, replace peptide.gro / peptide.xtc")
print("with your own topology and trajectory files.")
print("\nEnvironment ready ✓")


# Introduction to MDAnalysis: Analyzing Peptide Simulations
## Demonstration Walkthrough

This notebook is a **guided demonstration** of how to use MDAnalysis to analyse a molecular dynamics simulation of a peptide. Each section introduces a concept and shows it running on real simulation data, with commentary explaining the thought process and the chemistry behind each measurement.

**Files required:** Place `peptide.tpr` (GROMACS topology) and `peptide.xtc` (trajectory) in the same directory as this notebook before running.

| Section | Topic |
|---------|-------|
| 1 | Installation and imports |
| 2 | Loading a simulation with `Universe` |
| 3 | Selecting atoms with the selection language |
| 4 | Inspecting system properties |
| 5 | Iterating over trajectory frames |
| 6 | Radius of gyration |
| 7 | RMSD — structural drift over time |
| 8 | RMSF — per-residue flexibility |
| 9 | Distances and contact maps |
| 10 | Hydrogen bond analysis |
| 11 | A reusable analysis script |

---
## Section 1 — Installation and Imports

MDAnalysis is not part of the Python standard library, so it must be installed separately. The recommended way is through `conda` or `pip`.

```bash
# Using conda (recommended — handles all compiled dependencies)
conda install -c conda-forge mdanalysis mdanalysistests

# Or using pip
pip install MDAnalysis MDAnalysisTests
```

The cell below checks that MDAnalysis is installed and prints the version.

In [1]:
import MDAnalysis as mda
print("MDAnalysis version:", mda.__version__)

# Other packages used throughout this notebook
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# MDAnalysis analysis submodules (imported as needed later)
from MDAnalysis.analysis import rms, align
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
import MDAnalysis.lib.distances as distances

print("All imports successful!")

/opt/miniconda3/envs/plumed-masterclass-22/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MDAnalysis version: 2.9.0
All imports successful!


---
## Section 2 — Loading a Simulation: The `Universe`

The central object in MDAnalysis is the **`Universe`**. A `Universe` holds *everything* about a simulation:
- the **topology** (atom names, residue names, bonds, charges, masses)
- the **trajectory** (positions as a function of time)

You create a `Universe` by passing the topology file and trajectory file to `mda.Universe()`:

```python
u = mda.Universe("topology_file", "trajectory_file")
```

For GROMACS data, the topology is a `.tpr` file and the trajectory is an `.xtc` file.

### A note on coordinate units
MDAnalysis stores all distances in **Ångströms (Å)** and all times in **picoseconds (ps)**, regardless of what the simulation program used internally. You do not need to perform unit conversions for the quantities you will encounter in this notebook.

In [2]:
# Load the topology and trajectory into a Universe.
# Adjust the filenames below to match the files you have placed alongside this notebook.

TPR = "peptide.gro"   # GROMACS topology (GRO format — loaded from MDAnalysis test data in setup cell)
XTC = "peptide.xtc"   # GROMACS trajectory

u = mda.Universe(TPR, XTC)

print(f"Number of atoms   : {u.atoms.n_atoms}")
print(f"Number of residues: {u.atoms.n_residues}")
print(f"Number of frames  : {len(u.trajectory)}")
print(f"Timestep (ps)     : {u.trajectory.dt:.2f}")
print(f"Total time (ns)   : {u.trajectory.totaltime / 1000:.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'peptide.tpr'

The five lines of output tell us everything we need to know before starting an analysis: how many atoms and residues are in the box, how many frames were saved, the write interval (timestep), and the total simulated time. Checking these values first is a good habit — it catches mismatched file pairs before any real computation begins.

---
## Section 3 — Selecting Atoms

A `Universe` contains all atoms in the system — solute, solvent, ions, and everything else. Most analyses focus on a **subset** of atoms. MDAnalysis provides a human-readable **selection language** that should feel familiar if you have ever used VMD or CHARMM.

You call `u.select_atoms("selection string")` and get back an **`AtomGroup`** — an ordered collection of atoms on which you can compute properties.

### Common selection keywords

| Keyword | Selects | Example |
|---------|---------|--------|
| `protein` | All protein atoms | `"protein"` |
| `backbone` | N, CA, C, O backbone atoms | `"backbone"` |
| `resname ABC` | Residues named ABC | `"resname ALA"` |
| `resid 1:10` | Residue numbers 1–10 | `"resid 1:10"` |
| `name CA` | Atoms named CA | `"name CA"` |
| `not water` | Everything except water | `"not resname SOL"` |
| `and`, `or`, `not` | Boolean logic | `"protein and name CA"` |
| `around N sel` | Atoms within N Å of sel | `"around 5.0 resname LIG"` |

In [ ]:
# Select different subsets of the system

protein   = u.select_atoms("protein")
backbone  = u.select_atoms("backbone")
ca_atoms  = u.select_atoms("protein and name CA")
solvent   = u.select_atoms("resname SOL")   # GROMACS water residue name

print(f"Protein atoms  : {protein.n_atoms}")
print(f"Backbone atoms : {backbone.n_atoms}")
print(f"Cα atoms       : {ca_atoms.n_atoms}  (one per residue)")
print(f"Solvent atoms  : {solvent.n_atoms}")
print()
print("First 5 Cα residue names:", ca_atoms.resnames[:5])

The selection language reads almost like plain English, which makes it easy to construct and audit selections. Internally MDAnalysis compiles the selection string into a fast set-membership test, so even selections on systems with hundreds of thousands of atoms complete in milliseconds. The resulting `AtomGroup` is a *view* into the Universe — it always reflects the current frame's positions without any copying.

---
## Section 4 — Inspecting the System

An `AtomGroup` exposes atom-level information as **numpy arrays**. Since you learned about lists and loops in Part 1, numpy arrays should feel intuitive — they behave like lists but support fast element-wise math.

### Useful `AtomGroup` attributes

| Attribute | Type | Contents |
|-----------|------|----------|
| `.names` | `array of str` | Atom names (`CA`, `N`, `CB`, …) |
| `.resnames` | `array of str` | Residue names (`ALA`, `GLY`, …) |
| `.resids` | `array of int` | Residue sequence numbers |
| `.masses` | `array of float` | Atomic masses (amu) |
| `.charges` | `array of float` | Partial charges (e) |
| `.positions` | `(N,3) float array` | XYZ coordinates in Å at the **current frame** |
| `.n_atoms` | `int` | Number of atoms in the group |

In [ ]:
# Jump to the first frame and inspect Cα atom properties
u.trajectory[0]   # rewind to frame 0

print("Residue names:", ca_atoms.resnames)
print("Residue IDs  :", ca_atoms.resids)
print()
print("Cα positions at frame 0 (first 3 residues, Å):")
print(ca_atoms.positions[:3])
print()
print("Total mass of protein (amu):", protein.masses.sum().round(2))

Because `.positions` returns a live numpy array that is updated each time the trajectory advances, it is important to copy the array (with `.copy()`) if you need to store positions from a specific frame for later comparison. Here we are just printing, so no copy is needed.

---
## Section 5 — Iterating Over Frames

An MD trajectory is a sequence of **frames** — snapshots of atom positions separated by a fixed time interval (the trajectory write frequency). MDAnalysis exposes the trajectory as an iterable, so you can loop over it just like a Python list.

```python
for ts in u.trajectory:
    # ts is a Timestep object
    print(ts.frame, ts.time)   # frame index and time in ps
    # atom positions are updated automatically:
    pos = some_atom_group.positions
```

Each time the loop advances, MDAnalysis automatically updates the `.positions` of **all** `AtomGroup` objects so that they reflect the new frame — there is no need to pass the frame index explicitly.

You can also jump to a specific frame directly:
```python
u.trajectory[frame_index]
```

In [ ]:
# Collect the time (ps) and the x-coordinate of the first Cα atom across all frames

times  = []
x_traj = []

for ts in u.trajectory:
    times.append(ts.time)                     # simulation time in ps
    x_traj.append(ca_atoms.positions[0, 0])   # x-coord of first Cα (Å)

times  = np.array(times)
x_traj = np.array(x_traj)

print(f"Frames collected: {len(times)}")
print(f"Time range: {times[0]:.1f} – {times[-1]:.1f} ps")
print(f"x range for first Cα: {x_traj.min():.2f} – {x_traj.max():.2f} Å")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, x_traj, linewidth=0.8)
plt.xlabel("Time (ns)")
plt.ylabel("x-coordinate of first Cα (Å)")
plt.title("First Cα x-position over time")
plt.tight_layout()
plt.show()

The trajectory loop is the core of nearly every MDAnalysis script. Each iteration loads the next frame's coordinates from disk and updates all `AtomGroup.positions` in place. Keeping the loop body lightweight — appending a single float, not the full coordinate array — keeps memory use under control even for multi-microsecond trajectories.

---
## Section 6 — Center of Mass and Radius of Gyration

Two simple but informative quantities for a peptide are:

**Center of mass (CoM):**  
The mass-weighted average position of all atoms, analogous to the balance point of a physical object.

$$\vec{r}_{\text{CoM}} = \frac{\sum_i m_i \vec{r}_i}{\sum_i m_i}$$

**Radius of gyration (R_g):**  
The root-mean-square distance of atoms from the CoM. A compact, folded peptide has a small R_g; an extended, unfolded one has a large R_g.

$$R_g = \sqrt{\frac{\sum_i m_i |\vec{r}_i - \vec{r}_{\text{CoM}}|^2}{\sum_i m_i}}$$

MDAnalysis provides both as single-line method calls on any `AtomGroup`:
```python
com = protein.center_of_mass()   # returns (x, y, z) in Å
rg  = protein.radius_of_gyration()  # returns scalar in Å
```

In [ ]:
# Track radius of gyration over the trajectory

rg_values = []

for ts in u.trajectory:
    rg_values.append(protein.radius_of_gyration())

rg_values = np.array(rg_values)

print(f"Mean Rg   : {rg_values.mean():.2f} Å")
print(f"Std Rg    : {rg_values.std():.2f} Å")
print(f"Min Rg    : {rg_values.min():.2f} Å  (most compact)")
print(f"Max Rg    : {rg_values.max():.2f} Å  (most extended)")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, rg_values, linewidth=0.8, color='steelblue')
plt.axhline(rg_values.mean(), linestyle='--', color='red', label=f'Mean = {rg_values.mean():.2f} Å')
plt.xlabel("Time (ns)")
plt.ylabel("R$_g$ (Å)")
plt.title("Radius of Gyration over Time")
plt.legend()
plt.tight_layout()
plt.show()

Radius of gyration is a single-number summary of how compact the peptide is. Plotting it over time quickly reveals whether the peptide has collapsed into a compact state, remains extended, or transitions between the two. The mean and standard deviation are also useful summary statistics to report alongside RMSD in a methods section.

---
## Section 7 — RMSD: Measuring Structural Drift

**Root-Mean-Square Deviation (RMSD)** measures how much a structure has moved relative to a reference structure after optimal superposition. It is one of the most common quantities in MD analysis.

$$\text{RMSD}(t) = \sqrt{\frac{1}{N}\sum_{i=1}^{N} |\vec{r}_i(t) - \vec{r}_i^{\text{ref}}|^2}$$

where the sum is over the $N$ selected atoms and $\vec{r}_i^{\text{ref}}$ is the reference position of atom $i$ (usually the first frame or a known crystal structure).

**How to interpret RMSD:**
- An RMSD that quickly plateaus → the peptide reaches a stable conformation.
- An RMSD that keeps rising → the peptide is still drifting; the simulation may need to run longer.
- An RMSD that oscillates between two values → the peptide may be sampling two distinct states.

MDAnalysis provides `MDAnalysis.analysis.rms.RMSD` to handle the optimal rotation (superposition) automatically.

In [ ]:
# Compute the backbone RMSD relative to the first frame
# MDAnalysis needs to align structures before measuring deviation,
# otherwise global translation/rotation would inflate the number.

R = rms.RMSD(
    u,                              # Universe to analyse
    select="backbone",              # atoms used for fitting AND RMSD
    ref_frame=0                     # reference: frame 0
)
R.run()

# Results are in R.results.rmsd: columns are [frame, time(ps), RMSD(Å)]
rmsd_data = R.results.rmsd
rmsd_time = rmsd_data[:, 1] / 1000   # convert ps → ns
rmsd_vals = rmsd_data[:, 2]           # RMSD in Å

print(f"Final RMSD  : {rmsd_vals[-1]:.2f} Å")
print(f"Maximum RMSD: {rmsd_vals.max():.2f} Å at {rmsd_time[rmsd_vals.argmax()]:.3f} ns")

plt.figure(figsize=(7, 3))
plt.plot(rmsd_time, rmsd_vals, linewidth=0.9, color='navy')
plt.xlabel("Time (ns)")
plt.ylabel("Backbone RMSD (Å)")
plt.title("RMSD relative to frame 0")
plt.tight_layout()
plt.show()

The `rms.RMSD` class handles the least-squares superposition (rotation + translation) that must precede any RMSD measurement — otherwise global tumbling of the peptide in the box would dominate and inflate the values. The result array has three columns: frame index, simulation time in ps, and RMSD in Å. Dividing time by 1000 converts to ns for plotting.

---
## Section 8 — RMSF: Per-Residue Flexibility

While RMSD tells you how much the whole structure moves over **time**, **Root-Mean-Square Fluctuation (RMSF)** tells you how much each residue fluctuates around its **time-averaged position**. It is a per-atom (or per-residue) property rather than a per-frame one.

$$\text{RMSF}_i = \sqrt{\left\langle |\vec{r}_i(t) - \langle\vec{r}_i\rangle|^2 \right\rangle_t}$$

Large RMSF → flexible loop or terminus.  
Small RMSF → rigid secondary structure (helix, sheet).

RMSF is closely related to the **B-factor** (also called the temperature factor) reported in PDB X-ray structures:

$$B = \frac{8\pi^2}{3}\,\text{RMSF}^2$$

This connection lets you compare simulation flexibility directly to crystallographic data.

In [ ]:
# First align the trajectory to remove overall rotation/translation,
# then compute per-residue RMSF on the Cα atoms.

# Step 1: align all frames to the first frame (backbone fit)
aligner = align.AlignTraj(u, u, select="backbone", in_memory=True)
aligner.run()

# Step 2: compute RMSF on Cα atoms
rmsf_calc = rms.RMSF(ca_atoms)
rmsf_calc.run()

rmsf_values = rmsf_calc.results.rmsf  # one value per atom in ca_atoms

print("RMSF per Cα (Å):")
for resname, resid, r in zip(ca_atoms.resnames, ca_atoms.resids, rmsf_values):
    bar = '█' * int(r * 4)
    print(f"  {resname:3s} {resid:3d}: {r:5.2f} Å  {bar}")

plt.figure(figsize=(8, 3))
plt.bar(ca_atoms.resids, rmsf_values, color='mediumseagreen', edgecolor='none')
plt.xlabel("Residue number")
plt.ylabel("RMSF (Å)")
plt.title("Per-Residue Cα RMSF")
plt.tight_layout()
plt.show()

Aligning before computing RMSF is critical: without it, the per-atom fluctuations include contributions from overall rotation of the peptide, which can make every residue appear artificially flexible. After alignment the RMSF reflects only internal motion. Bar charts with residue number on the x-axis are the conventional way to present RMSF — regions of high flexibility (loops, termini) stand out immediately.

---
## Section 9 — Distances and Contacts

Computing distances between atoms is fundamental to structural biology. Common applications include:
- Checking whether a salt bridge forms and persists
- Monitoring hydrogen bond distances
- Counting native contacts to quantify folding

MDAnalysis offers two main approaches:

**1. Manual numpy distance** (fast for a single pair):
```python
d = np.linalg.norm(atom1.position - atom2.position)
```

**2. `MDAnalysis.lib.distances.dist()`** (handles periodic boundary conditions correctly, preferred for bulk properties):
```python
from MDAnalysis.lib import distances
d_array = distances.dist(group_a, group_b)   # returns (3, N) array: [frame_dist, ...]
```

For **contact maps** (is residue i close to residue j?), you can use `distances.self_distance_array()` on Cα positions.

In [ ]:
# Monitor the distance between the Cα of residue 1 and residue N over the trajectory.
# This is the end-to-end Cα distance (complement to the end-to-end distance above).

from MDAnalysis.lib.distances import dist as mda_dist

ca_first_ag = u.select_atoms("name CA and resid 1")
ca_last_ag  = u.select_atoms(f"name CA and resid {ca_atoms.resids[-1]}")

dist_over_time = []
for ts in u.trajectory:
    result = mda_dist(ca_first_ag, ca_last_ag, box=ts.dimensions)
    dist_over_time.append(result[2][0])   # result[2] is the distance array

dist_over_time = np.array(dist_over_time)

print(f"Mean end-to-end (Cα) distance: {dist_over_time.mean():.2f} ± {dist_over_time.std():.2f} Å")

The `dist()` function from `MDAnalysis.lib.distances` is preferred over a hand-written `np.linalg.norm` call when periodic boundary conditions matter, because it applies the minimum-image convention automatically using the box dimensions stored in `ts.dimensions`. For a single end-to-end distance the difference may be negligible, but for distances across the box it can be substantial.

In [ ]:
# A contact map shows which pairs of residues are in spatial proximity.
# Conventionally, two residues are 'in contact' if their Cα atoms are within 8 Å.

from MDAnalysis.lib.distances import self_distance_array

u.trajectory[-1]   # jump to last frame

# self_distance_array returns the upper-triangle of the pairwise distance matrix
ca_pos  = ca_atoms.positions
n_res   = len(ca_pos)
dists   = self_distance_array(ca_pos.astype(np.float32))

# Reconstruct full symmetric matrix from condensed upper-triangle
contact_matrix = np.zeros((n_res, n_res))
idx = np.triu_indices(n_res, k=1)
contact_matrix[idx]      = dists
contact_matrix           = contact_matrix + contact_matrix.T
contact_map              = contact_matrix < 8.0   # True if within 8 Å

plt.figure(figsize=(5, 4))
plt.imshow(contact_map, cmap='Blues', origin='lower',
           extent=[ca_atoms.resids[0], ca_atoms.resids[-1],
                   ca_atoms.resids[0], ca_atoms.resids[-1]])
plt.colorbar(label='Contact (< 8 Å)')
plt.xlabel("Residue i")
plt.ylabel("Residue j")
plt.title("Cα Contact Map (last frame)")
plt.tight_layout()
plt.show()

A contact map is a two-dimensional representation of the full pairwise distance matrix. The diagonal is trivially zero (every atom is at distance zero from itself), so it is usually masked out. Off-diagonal blocks that are dark blue indicate structurally proximate residues — helices appear as bands parallel to the diagonal at a spacing of 3–4 residues, while β-sheet hydrogen-bonding partners appear as off-diagonal patches.

---
## Section 10 — Hydrogen Bond Analysis

Hydrogen bonds are critical to the secondary structure of peptides. An α-helix, for example, is stabilized by hydrogen bonds between the backbone N–H of residue i and the backbone C=O of residue i−4.

MDAnalysis defines a hydrogen bond by three geometric criteria:

| Criterion | Default threshold |
|-----------|-------------------|
| Donor–Acceptor distance | ≤ 3.5 Å |
| Donor–H–Acceptor angle | ≥ 120° |

The `HydrogenBondAnalysis` class automatically identifies donors and acceptors from atom names (O, N for acceptors; O–H, N–H for donors) and screens every frame of the trajectory.

> **Note on GROMACS .tpr files:** `.tpr` files do not always store explicit hydrogen atoms (some force fields use united-atom representations). If `HydrogenBondAnalysis` finds no H atoms, you may need to use a `.gro` or `.pdb` file as topology, or pass `hydrogens_sel` explicitly.

In [ ]:
# Identify backbone hydrogen bonds within the protein

hbonds = HydrogenBondAnalysis(
    universe      = u,
    donors_sel    = "protein and name N",       # backbone N as donors
    hydrogens_sel = "protein and name H",       # backbone H
    acceptors_sel = "protein and name O",       # backbone O as acceptors
    d_a_cutoff    = 3.5,                        # donor–acceptor distance cutoff (Å)
    d_h_a_angle   = 120.0                       # minimum D–H···A angle (degrees)
)
hbonds.run()

# Count hydrogen bonds per frame
hb_count_per_frame = []
for frame_hbs in hbonds.results.hbonds:
    # frame_hbs is a list of [frame, donor_idx, H_idx, acceptor_idx, dist, angle]
    hb_count_per_frame.append(len(frame_hbs))

hb_count_per_frame = np.array(hb_count_per_frame)
print(f"Mean backbone H-bonds: {hb_count_per_frame.mean():.1f} ± {hb_count_per_frame.std():.1f}")

plt.figure(figsize=(7, 3))
plt.plot(times / 1000, hb_count_per_frame, linewidth=0.8, color='firebrick')
plt.xlabel("Time (ns)")
plt.ylabel("Number of backbone H-bonds")
plt.title("Backbone Hydrogen Bonds over Time")
plt.tight_layout()
plt.show()

The geometric criteria used here — donor–acceptor distance ≤ 3.5 Å and D–H···A angle ≥ 120° — are the most commonly used defaults in the field. An α-helix stabilised by i→i+4 backbone H-bonds should show a relatively constant count of ~(N_residues − 4) H-bonds throughout the simulation. A drop in this count often signals partial unfolding.

---
## Section 11 — A Reusable Analysis Script

Throughout this notebook you have written individual analysis snippets. In practice, you will want to package these into reusable functions and run them from the command line. The cell below shows how to organise all the analyses you have learned into a single, well-documented Python function.

This section is **read-and-run** — study the structure and then customise it for your own project.

In [ ]:
# --- Complete reusable analysis function ---

def analyze_peptide(tpr_path: str, xtc_path: str, outdir: str = ".") -> dict:
    """
    Run a standard set of analyses on a GROMACS peptide simulation.

    Parameters
    ----------
    tpr_path : str
        Path to the GROMACS .tpr topology file.
    xtc_path : str
        Path to the GROMACS .xtc trajectory file.
    outdir : str
        Directory where result plots will be saved.

    Returns
    -------
    dict
        Dictionary containing the computed arrays:
        'time_ns', 'rg', 'rmsd', 'rmsf', 'hbond_count'
    """
    import os
    os.makedirs(outdir, exist_ok=True)

    # --- Load universe ---
    u = mda.Universe(tpr_path, xtc_path)
    protein   = u.select_atoms("protein")
    backbone  = u.select_atoms("backbone")
    ca_atoms  = u.select_atoms("protein and name CA")

    # --- Time axis ---
    times_ns = np.array([ts.time / 1000 for ts in u.trajectory])

    # --- Radius of gyration ---
    rg_vals = []
    for ts in u.trajectory:
        rg_vals.append(protein.radius_of_gyration())
    rg_vals = np.array(rg_vals)

    # --- RMSD ---
    R = rms.RMSD(u, select="backbone", ref_frame=0)
    R.run()
    rmsd_vals = R.results.rmsd[:, 2]

    # --- RMSF (after alignment) ---
    aligner = align.AlignTraj(u, u, select="backbone", in_memory=True)
    aligner.run()
    rmsf_calc = rms.RMSF(ca_atoms)
    rmsf_calc.run()
    rmsf_vals = rmsf_calc.results.rmsf

    # --- Hydrogen bonds ---
    hbonds = HydrogenBondAnalysis(
        universe=u,
        donors_sel="protein and name N",
        hydrogens_sel="protein and name H",
        acceptors_sel="protein and name O",
    )
    hbonds.run()
    hb_counts = np.array([len(f) for f in hbonds.results.hbonds])

    # --- Plots ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(times_ns, rg_vals, linewidth=0.9, color='steelblue')
    axes[0, 0].set(xlabel='Time (ns)', ylabel='Rg (Å)', title='Radius of Gyration')

    axes[0, 1].plot(times_ns, rmsd_vals, linewidth=0.9, color='navy')
    axes[0, 1].set(xlabel='Time (ns)', ylabel='RMSD (Å)', title='Backbone RMSD')

    axes[1, 0].bar(ca_atoms.resids, rmsf_vals, color='mediumseagreen', edgecolor='none')
    axes[1, 0].set(xlabel='Residue', ylabel='RMSF (Å)', title='Per-Residue RMSF')

    axes[1, 1].plot(times_ns, hb_counts, linewidth=0.8, color='firebrick')
    axes[1, 1].set(xlabel='Time (ns)', ylabel='H-bonds', title='Backbone H-Bonds')

    plt.tight_layout()
    fig.savefig(f"{outdir}/peptide_summary.png", dpi=150)
    plt.show()
    print(f"Summary figure saved to {outdir}/peptide_summary.png")

    return {
        'time_ns'     : times_ns,
        'rg'          : rg_vals,
        'rmsd'        : rmsd_vals,
        'rmsf'        : rmsf_vals,
        'hbond_count' : hb_counts,
    }


# --- Run the full analysis ---
results = analyze_peptide(TPR, XTC, outdir="peptide_analysis_output")

print("\n=== Summary ===")
print(f"Mean Rg    : {results['rg'].mean():.2f} Å")
print(f"Mean RMSD  : {results['rmsd'].mean():.2f} Å")
print(f"Mean H-bonds: {results['hbond_count'].mean():.1f}")

Packaging the analysis into a single function with clear parameter names and a docstring makes it trivial to reuse across projects. The function returns a dictionary so callers can access individual arrays by name, pipe them into further analysis, or save them with `numpy.savez`. The four-panel summary figure gives a quick visual health-check of the simulation before committing time to deeper analysis.

---
## Summary and Next Steps

You have worked through a complete MDAnalysis workflow for peptide simulations:

1. **Loading** a `Universe` from GROMACS `.tpr` + `.xtc` files
2. **Selecting** atom subsets with human-readable selection strings
3. **Inspecting** atom attributes (names, masses, positions)
4. **Iterating** over trajectory frames
5. **Computing** global structural properties (Rg, end-to-end distance)
6. **Measuring** structural drift with RMSD
7. **Quantifying** flexibility with RMSF and B-factors
8. **Analysing** pairwise distances and native contacts
9. **Identifying** hydrogen bonds and their persistence
10. **Packaging** everything into a reusable function

**Where to go next:**
- **Dihedral angles (Ramachandran plots):** `MDAnalysis.analysis.dihedrals`
- **Secondary structure:** use `MDAnalysis` with `mdtraj` or DSSP for per-frame secondary structure
- **Solvent-accessible surface area (SASA):** `MDAnalysis.analysis.hydrophobicanalysis` or Freesasa
- **Water structure:** analyse solvent shells around the peptide
- **Dimensionality reduction (PCA/TICA):** `MDAnalysis.analysis.pca` for identifying dominant motions

**Useful references:**
- MDAnalysis documentation: https://docs.mdanalysis.org
- MDAnalysis User Guide: https://userguide.mdanalysis.org
- MDAnalysis publication: Michaud-Agrawal et al., *J. Comput. Chem.* 2011, 32, 2319–2327